---
title: "Module 2 - Lab 1"
subtitle: "SEC HTML Section Extraction and Corpus Validation"
number-sections: true
date: "2026-02-01"
date-modified: today
date-format: long
engine: jupyter
format:
  html:
    toc: true
    toc-depth: 3
    code-overflow: wrap
bibliography: ../references.bib
csl: ../mis-quarterly.csl
execute:
  echo: true
  eval: false
  warning: false
  message: false
categories: ['M02:', 'Lab']
description: "Hands-on lab for Module 2 focused on extracting, validating, and chunking reusable SEC filing text."
---

## Overview

In this lab you will turn messy SEC HTML into a **validated, reusable text corpus** that later Module 2 work can build on.

Starting from raw filings, you will:

1. discover and load SEC HTML files
2. clean HTML into normalized plain text
3. extract Item 7, Item 7A, and Item 8 sections
4. validate extraction quality with summary checks and spot reviews
5. chunk the extracted sections into retrieval-ready text windows
6. save practice artifacts that mirror the structure used later in the module

::: {.callout-note}
All code cells have `eval: false` by default. Run them in order inside Jupyter or Google Colab.
:::

## How This Lab Connects

- `M02_T1` introduces the extraction helpers and section-boundary logic in a step-by-step tutorial format.
- This lab asks you to **validate** that workflow on a small filing sample and save clean downstream artifacts.
- `M02_T2` and `M02_Lab2` use chunked SEC text for sparse vs dense semantic analysis.
- `M02_A` extends the same habits to a graded business task, but students must still use their own companies and executed notebook evidence.

::: {.callout-important}
This lab is preparatory, not a copy-paste answer key for the assignment. The goal is to practice the workflow on a smaller, instructor-friendly dataset before applying similar ideas in your own graded work.
:::

## Recommended Notebook Pattern

Keep the notebook organized as a reproducible data pipeline:

1. Setup and path configuration
2. File discovery
3. HTML cleaning helpers
4. Section extraction
5. Validation diagnostics
6. Chunking
7. Artifact export

## Validation Standard

For each major stage, show at least one check such as:

- number of files processed
- missing-section counts
- section-length summary statistics
- one manually reviewed extraction example
- chunk-count summary by section
- one sample of saved chunk text

If this lab requires a short write-up in addition to the notebook, format that written deliverable in **APA 7**.

## Setup

In [ ]:
#| eval: false
import json
import random
import re
import unicodedata
from collections import Counter
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## Discover SEC HTML Files

Use a small sample first so you can debug extraction before scaling.

In [ ]:
#| eval: false
DATA_DIR = Path("../data/SEC-10K-2024-HTML")
OUTPUT_DIR = Path("../data/outputs/module2_lab1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

all_html_files = sorted(DATA_DIR.rglob("*.html")) + sorted(DATA_DIR.rglob("*.htm"))
print(f"Total HTML files found: {len(all_html_files)}")
all_html_files[:5]

In [ ]:
#| eval: false
def select_html_files(files, sample_n: Optional[int] = 10, random_sample: bool = False, seed: int = 42):
    if sample_n is None:
        return files
    sample_n = min(sample_n, len(files))
    if random_sample:
        rng = random.Random(seed)
        return sorted(rng.sample(files, sample_n))
    return files[:sample_n]


html_files = select_html_files(all_html_files, sample_n=10, random_sample=False, seed=SEED)
print(f"Files selected for this lab: {len(html_files)}")
html_files[:3]

## Clean HTML into Plain Text

In [ ]:
#| eval: false
def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\xa0", " ")
    text = text.replace("&nbsp;", " ")
    text = re.sub(r"\r", "\n", text)
    text = re.sub(r"\n{2,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


def html_to_text(html: str) -> str:
    soup = BeautifulSoup(html, "lxml")
    for tag in soup(["script", "style", "ix:header", "header", "footer"]):
        tag.decompose()
    return normalize_text(soup.get_text(separator="\n"))

## Extract Item 7, Item 7A, and Item 8

These are the same SEC sections most relevant to later chunk-level semantic analysis.

In [ ]:
#| eval: false
ITEM_PATTERNS = {
    "item_7": re.compile(r"(?im)^\s*item\s*7[\.\-:\s]*\b(?!a\b)(?:management'?s?\s+discussion\b|md&a\b)?"),
    "item_7a": re.compile(r"(?im)^\s*item\s*7a[\.\-:\s]*\b"),
    "item_8": re.compile(r"(?im)^\s*item\s*8[\.\-:\s]*\b"),
}


def extract_section_by_bounds(text: str, start_pattern: re.Pattern, end_patterns: list[re.Pattern],
                              min_chars: int = 1000, max_chars: int = 2_000_000) -> Optional[str]:
    start_positions = [m.start() for m in start_pattern.finditer(text)]
    if not start_positions:
        return None

    end_positions = []
    for pat in end_patterns:
        end_positions.extend(m.start() for m in pat.finditer(text))
    end_positions = sorted(set(end_positions))

    candidates = []
    for start in start_positions:
        valid_ends = [end for end in end_positions if end > start]
        if not valid_ends:
            continue
        end = valid_ends[0]
        span_len = end - start
        if min_chars <= span_len <= max_chars:
            candidates.append((start, end, span_len))

    if not candidates:
        return None

    best_start, best_end, _ = max(candidates, key=lambda x: x[2])
    return text[best_start:best_end].strip()


def extract_tail_section(text: str, start_pattern: re.Pattern, min_chars: int = 1000) -> Optional[str]:
    matches = list(start_pattern.finditer(text))
    if not matches:
        return None
    start = matches[-1].start()
    section = text[start:].strip()
    return section if len(section) >= min_chars else None


def extract_target_sections(text: str) -> dict[str, Optional[str]]:
    item_7 = extract_section_by_bounds(
        text=text,
        start_pattern=ITEM_PATTERNS["item_7"],
        end_patterns=[ITEM_PATTERNS["item_7a"], ITEM_PATTERNS["item_8"]],
        min_chars=1000,
    )
    item_7a = extract_section_by_bounds(
        text=text,
        start_pattern=ITEM_PATTERNS["item_7a"],
        end_patterns=[ITEM_PATTERNS["item_8"]],
        min_chars=500,
    )
    item_8 = extract_tail_section(text, ITEM_PATTERNS["item_8"], min_chars=1000)
    return {"item_7": item_7, "item_7a": item_7a, "item_8": item_8}

## Build the Section-Level Corpus

In [ ]:
#| eval: false
records = []

for path in html_files:
    raw_html = path.read_text(encoding="utf-8", errors="ignore")
    clean_text = html_to_text(raw_html)
    sections = extract_target_sections(clean_text)

    records.append({
        "file_name": path.name,
        "file_path": str(path),
        "clean_char_count": len(clean_text),
        "item_7": sections["item_7"],
        "item_7a": sections["item_7a"],
        "item_8": sections["item_8"],
        "item_7_len": len(sections["item_7"]) if sections["item_7"] else 0,
        "item_7a_len": len(sections["item_7a"]) if sections["item_7a"] else 0,
        "item_8_len": len(sections["item_8"]) if sections["item_8"] else 0,
        "missing_item_7": sections["item_7"] is None,
        "missing_item_7a": sections["item_7a"] is None,
        "missing_item_8": sections["item_8"] is None,
    })

sections_df = pd.DataFrame(records)
print(sections_df.shape)
sections_df.info()
sections_df.describe(include="all")
sections_df.head()

## Validate Extraction Quality

In [ ]:
#| eval: false
missing_summary = sections_df[["missing_item_7", "missing_item_7a", "missing_item_8"]].sum()
missing_summary

In [ ]:
#| eval: false
length_cols = ["item_7_len", "item_7a_len", "item_8_len"]
sections_df[length_cols].describe()

In [ ]:
#| eval: false
sections_df[length_cols].plot(kind="box", figsize=(8, 4), title="Section Length Distribution")
plt.ylabel("Characters")
plt.show()

In [ ]:
#| eval: false
review_row = sections_df.loc[~sections_df["missing_item_7"], ["file_name", "item_7"]].head(1)
review_row

## Convert Sections into Chunk-Level Text

Later module work usually happens at the chunk level rather than the whole-section level.

In [ ]:
#| eval: false
def split_sentences(text: str) -> list[str]:
    if not isinstance(text, str) or not text.strip():
        return []
    parts = re.split(r"(?<=[.!?])\s+", text.strip())
    return [p.strip() for p in parts if p.strip()]


def build_sentence_windows(sentences: list[str], window_size: int = 4, step: int = 2) -> list[str]:
    windows = []
    for start in range(0, max(len(sentences) - window_size + 1, 1), step):
        window = sentences[start:start + window_size]
        if window:
            windows.append(" ".join(window))
    return windows

In [ ]:
#| eval: false
chunk_rows = []
chunk_id = 1

for _, row in sections_df.iterrows():
    for item in ["item_7", "item_7a", "item_8"]:
        section_text = row[item]
        sentences = split_sentences(section_text)
        windows = build_sentence_windows(sentences, window_size=4, step=2)

        for idx, chunk_text in enumerate(windows, start=1):
            chunk_rows.append({
                "chunk_id": f"lab1_{chunk_id:05d}",
                "file_name": row["file_name"],
                "item": item.replace("_", " ").title(),
                "chunk_index": idx,
                "chunk_text": chunk_text,
                "char_count": len(chunk_text),
                "token_count": len(chunk_text.split()),
            })
            chunk_id += 1

chunks_df = pd.DataFrame(chunk_rows)
print(chunks_df.shape)
chunks_df.info()
chunks_df.describe(include="all")
chunks_df.head()

In [ ]:
#| eval: false
chunks_df.groupby("item")["chunk_id"].count().plot(
    kind="bar",
    title="Chunk Count by Section",
    figsize=(7, 4)
)
plt.ylabel("Number of chunks")
plt.show()

## Build a Lightweight Vocabulary Artifact

This mirrors the idea of saving reusable downstream artifacts without pretending this lab output is the assignment deliverable.

In [ ]:
#| eval: false
token_counter = Counter()
for text in chunks_df["chunk_text"]:
    token_counter.update(re.findall(r"[a-z]+", text.lower()))

vocab = sorted(token_counter.keys())
print(f"Vocabulary size: {len(vocab):,}")
vocab[:20]

## Save Practice Artifacts

In [ ]:
#| eval: false
sections_path = OUTPUT_DIR / "lab1_sections.csv"
chunks_path = OUTPUT_DIR / "lab1_chunks.csv"
vocab_path = OUTPUT_DIR / "lab1_vocab.json"

sections_df.to_csv(sections_path, index=False)
chunks_df.to_csv(chunks_path, index=False)
with open(vocab_path, "w", encoding="utf-8") as f:
    json.dump(vocab, f, indent=2)

print("Saved:")
print(" ", sections_path)
print(" ", chunks_path)
print(" ", vocab_path)

## Lab Questions

Answer the following questions in a separate `.md` or `.docx` file unless your instructor asks for embedded answers.

**Q1. Extraction Coverage**  
How many sampled filings were missing Item 7, Item 7A, or Item 8? Which section appeared hardest to capture reliably, and what filing-format issue seems most responsible?

**Q2. Section-Length Diagnostics**  
What do the section-length summaries suggest about extraction quality? Identify one suspiciously short or long section and explain whether it looks like a true filing characteristic or a parsing problem.

**Q3. Chunking Design**  
How did your `window_size` and `step` choices affect the number of chunks and average chunk length? What tradeoff do you see between context preservation and retrieval granularity?

**Q4. Artifact Reuse**  
Why is it useful to save separate section-level, chunk-level, and vocabulary artifacts instead of repeatedly rebuilding them from raw HTML?

**Q5. Business Interpretation**  
If a research team wanted to build semantic search over management discussion, why would chunk-level validation matter before training any embedding model or using an LLM?

## AI Use and Share Links {.unnumbered}

If generative AI materially supports your work for this lab, include an AI disclosure appendix or separate AI disclosure document if your instructor requests one. Include the complete prompt(s), relevant output excerpt(s), validation steps, and direct shared chat link(s) when available.

![](../M0/M0_lecture01_figures/chatgpt-share.jpg){width="80%" fig-align="center"}

![](../M0/M0_lecture01_figures/gemini-share-conversation.png){width="80%" fig-align="center"}

When possible, use **one continuous chat** for the activity so the full reasoning trail can be reviewed in one place. If you used multiple chats, share **all chats directly related** to the work.

Tools such as **ChatGPT**, **Claude**, and **Gemini** support direct chat sharing. If sharing or export is not available, include copied prompt/output evidence or screenshots instead. Because shared links may be viewable by anyone with the link, do not include confidential, personal, or restricted information in those chats.